## Returns and Rates

### 1.Holding Period Return
#### Simple HPR = (Income + (price at end - price at begining)) / price at begining

In [1]:
def simHPR(div, initial_price, current_prince, n_shares):
    i_price = n_shares * initial_price
    c_price = n_shares * current_prince
    return ((div + (c_price - i_price))/i_price) * 100

print(f"{simHPR(51.55, 34.50, 30.50, 100)}%")

-10.1%


#### HPR over Multiple Time: Since every period have there specific invested wealth so, we divide them in period.
##### Total HPR (1+R) = Product of ((1+R1)*(1+R2)..)

In [2]:
import numpy as np
def multiHPR(returns):
    returns = np.array(returns)
    return ((np.prod(list(map(lambda x: 1 + (x / 100), returns)))) - 1)*100

print(f"{multiHPR([10,20,-10])}%")


18.80000000000002%


### 2. Arithmetic Mean Return: Simply average of periodic return

In [3]:
def AMR(returns):
    return np.average(returns)

print(f"{AMR(np.array([10,20,-10]))}%")

6.666666666666667%


#### But AMR is does not show actually grew, as it does noyt coumpounding, its just averaging and tell us the expected return for a single future period, hence E(R) ~ 6.67%

In [4]:
print(f"{AMR(np.array([10,20,-10])) * 3} is {AMR(np.array([10,20,-10])) * 3 == multiHPR([10,20,-10])} to {multiHPR([10,20,-10])}")

20.0 is False to 18.80000000000002


In [5]:
print("AMR:",AMR(np.array([50,-50])))
print("HPR",multiHPR(np.array([50,-50])))
print("hence returns must be compounded when measring actual wealth growth")

AMR: 0.0
HPR -25.0
hence returns must be compounded when measring actual wealth growth


#### 3. Geometric Mean Return: Tells us Compund Growth Rate across multiple period. Gives invested experience
##### suppose n periodic return , GMR = ((product((1+R)))^1/n )-1
##### GMR = ((1+HPR)^(1/n)) -1

In [6]:
def GMR(returns):
    return ((np.prod(list(map(lambda x : (x/100) + 1, returns)))) ** (1/len(returns)) - 1) * 100

GMR(np.array([50,-50]))


-13.397459621556141

In [7]:
print("AMR:",AMR(np.array([50,-50])))
print("HPR",multiHPR(np.array([50,-50])))
print("GMR:",GMR(np.array([50,-50])))

AMR: 0.0
HPR -25.0
GMR: -13.397459621556141


In [8]:
axis_price = [80.49, 110.51, 108.93, 104.81] # Oct 23 to oct 26
shares = 100
# 1st way
axis_returns = [simHPR(0, curr, next, 1) for curr, next in zip(axis_price, axis_price[1:]) ]
axis_returns

#2nd way
np_array_axis = np.array([80.49, 110.51, 108.93, 104.81])
axis_returns = simHPR(0, np_array_axis[:-1], np_array_axis[1:], 1)
axis_returns

print("AMR:",AMR(axis_returns))
print("HPR",multiHPR(axis_returns))
print("GMR:",GMR(axis_returns))


AMR: 10.694859411444867
HPR 30.214933532115793
GMR: 9.199403118373528


### Harmonic Return, when we want avverage rates / multipliers. Harmonic averaging gives less weights to large observation


In [9]:
def harmonic_return(returns):
    return len(returns) / (np.sum(list(map(lambda x: 1/x, returns))))

print("HR:",harmonic_return(axis_returns))
print("HR:",harmonic_return(np.array([50,-50])))
print("HR:",harmonic_return(np.array([1,2,3,4,5,6,1000])))

HR: -3.2016684248292
HR: inf
HR: 2.855977152182783


C:\Users\Akshay\AppData\Local\Temp\ipykernel_16128\2506196778.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  return len(returns) / (np.sum(list(map(lambda x: 1/x, returns))))


##### AMR = Average , 
##### GMR= Compound Growth, 
##### HR = rates/ratios/multiples
##### Relationship : AMR * HR = G^2

In [10]:
axis_amr = AMR(np.array([1,2,3,4,5,6,1000]))
axis_harmonic = harmonic_return(np.array([1,2,3,4,5,6,1000]))
axis_gmr = GMR(np.array([1,2,3,4,5,6,1000]))
print(axis_amr, axis_harmonic, axis_gmr)
print(axis_amr * axis_harmonic, axis_gmr ** 2)

145.85714285714286 2.855977152182783 45.052663693247766
416.56466748266024 2029.7425058568854


### For extreme Outliers:
#### 1. Trimmed Mean: Remove specified percentage of lowest and highest observation then AMR wit remaining ones

In [11]:
returns = np.array([1,2,3,4,5,6,7,8,9,100])
amr = AMR(returns)
print("AMR:", amr)
trimmed_amr = AMR(returns[1:-1])
print("Trimmed AMR:", trimmed_amr)

AMR: 14.5
Trimmed AMR: 5.5


#### 2. Winsorized Mean: similar but dont throw observation away instead replace them with less extreme value

In [12]:
returns[0], returns[-1] = returns[1], returns[-2]
win_amr = AMR(returns)
print("Winsorized AMR:", win_amr)

Winsorized AMR: 5.5


## Money Weighted Return and Time Weighted Return
##### Intution: What happenes when Money enters and leaves a portfolio during the measurement period
##### AMR and GMR: dont account for the timing and size of investor cash flow
##### MWR: What return did the I actually earn on my money. Investor Experience
##### TWR: How well did the PM perform on the portfolio, independent of cash flow timing and size. Manager Performance

In [13]:
from scipy.optimize import brentq

def npv(r):
    return (
        -100000
        - 50000 / (1 + r)
        + 20000 / (1 + r)**2
        + 150000 / (1 + r)**3
    )

r = brentq(npv, -0.99, 10)

print(r)
print(f"MWR = {r:.2%}")

0.05027053105170753
MWR = 5.03%


In [16]:
def money_weighted_return(cash_flows):
    def npv(r):
        return sum(
            cf / (1 + r) ** t
            for t, cf in enumerate(cash_flows)
        )

    return brentq(npv, -0.9999, 10)

In [17]:
# cash_flows = [-100000, -50000, 20000, 150000]
cash_flows = [-200, -220, 480]

mwr = money_weighted_return(cash_flows)

print(f"MWR: {mwr:.2%}")

MWR: 9.39%


In [ ]:
import pandas as pd
df = pd.DataFrame({
    "Date": ["2026-01-01", "2026-02-01", "2026-03-01", "2026-04-01"],
    "Portfolio Value": [100, 120, 150, 140],
    "Cash Flows": [0, -20, 30, 0],
    "Dividends": [0, 2, 0, 1]
})
def HPR(initial_value, current_value, dividend):
    return (current_value - initial_value + dividend) / initial_value
def simpleTWR(returns):
    df["PF_After_Cashflow"] = df["Portfolio Value"] - df["Cash Flows"]
    periodic_returns = [HPR(initial_value, current_value, dividend) for initial_value, current_value, dividend in zip(df["PF_After_Cashflow"][:-1], df["Portfolio Value"][1:], df["Dividends"][1:])]
    return 